# Model pipeline: work in progress

Ongoing notebook: from the raw data to a first validated model, before cross-validation.

**Pipeline**

| Step | What | Status |
|---|---|---|
| 0 | Load the data (cached `.npy`), labels to {0, 1} | ok |
| 1 | Take a 50k-row random subsample | ok |
| 2 | Split the subsample 80% training / 20% validation | ok |
| 3 | Preprocess: **fit + transform + bias on the training part**, then **transform + bias on the validation part** | ok |
| 4 | Train L2-regularized logistic regression on the training part |ok  |
| 5 | Predict on the validation part and analyse (probabilities → best threshold → F1) |  F1 ≈ 0.45, threshold ≈ 0.25 |
| next | Stratified k-fold cross-validation, then experiments |  |


### Setup

In [ ]:
import sys
sys.path.insert(0, "..")   # also look in the parent folder (the repo root)

import numpy as np
import helpers
import preprocessing
import implementations

data_path = "../data"

## Functions

**Metric**
- **F1 score.** Each prediction is a true positive (TP), false positive (FP), false negative (FN) or true
  negative (TN). Precision = TP / (TP + FP), recall = TP / (TP + FN), and F1 is their harmonic mean.
  F1 ignores the true negatives, which suits our imbalanced data: predicting "healthy" for everyone gives
  TP = 0, so F1 = 0.
- **Best threshold.** The model outputs probabilities. With ~9% positives, the default cut at 0.5 predicts
  almost nobody as positive, so we try thresholds between 0 and 1 and keep the one with the best F1.

**Data**
- `subsample`: random distinct rows, x and y selected with the same indices so they stay aligned.
- `split_data`: shuffles the row indices and cuts them into a training part and a validation part.
- `prepare_features`: step 3 (fit on training, transform both, add bias) in a single call. Not used below,
  where step 3 is written out; it will be used in the cross-validation loop.

In [ ]:
def f1_score(y_true, y_pred):
    """
    F1 score of binary predictions.

    Args:
        y_true (np.array): true labels in {0, 1}, shape (N,)
        y_pred (np.array): predicted labels in {0, 1}, shape (N,)

    Returns:
        f1 (float): F1 score, 0.0 if there is no true positive (also avoids dividing by zero)
    """
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    if TP == 0:
        return 0.0

    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    f1 = 2 * (precision*recall) / (precision+recall)

    return f1


def best_threshold(y_true, scores):
    """
    Find the threshold on the probability scores that gives the best F1.

    Args:
        y_true (np.array): true labels in {0, 1}, shape (N,)
        scores (np.array): predicted probabilities in [0, 1] (sigmoid of tx @ w), shape (N,)

    Returns:
        best_t (float): threshold with the best F1 (predict 1 if score >= best_t)
        best_f1 (float): F1 obtained with that threshold
    """
    thresholds = np.linspace(0, 1, 101) # different thresholds between 0 and 1 (steps of 0.01)

    f1_scores = []

    for t in thresholds:
        y_pred = (scores>=t).astype(int)
        f1 = f1_score(y_true, y_pred)

        f1_scores.append(f1)

    best_idx = np.argmax(f1_scores)

    best_t = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    return best_t, best_f1


def subsample(x_train, y_train, size, rng):
    """
    Random subsample of the rows.

    Args:
        x_train (np.array): features, shape (N, D)
        y_train (np.array): labels, shape (N,)
        size (int): number of rows to keep
        rng (np.random.Generator): random generator, e.g. np.random.default_rng(42)

    Returns:
        x_sub (np.array): `size` distinct random rows of x_train, shape (size, D)
        y_sub (np.array): the matching labels, shape (size,)
    """
    idx = rng.choice(len(y_train), size=size, replace=False)  # `size` distinct row numbers in 0..N-1 - no replace avoid duplication

    return x_train[idx], y_train[idx]   # same idx for x and y: rows stay aligned


def split_data(x, y, ratio, rng):
    """
    Random split into a training part and a validation part.

    Args:
        x (np.array): features, shape (N, D)
        y (np.array): labels, shape (N,)
        ratio (float): share of the rows used for training (e.g. 0.8)
        rng (np.random.Generator): random generator

    Returns:
        x_tr, y_tr (np.array): training rows and labels (ratio * N rows)
        x_val, y_val (np.array): validation rows and labels (the rest)
    """
    cut = int(ratio*len(y))

    idx = rng.permutation(len(y))   # shuffled row numbers
    idx_tr = idx[:cut]
    idx_val = idx[cut:]

    x_tr, y_tr   = x[idx_tr], y[idx_tr]
    x_val, y_val = x[idx_val], y[idx_val]

    return x_tr, y_tr, x_val, y_val


def prepare_features(x_tr, x_val, names):
    """
    Fit the preprocessing on the training part only, transform both parts, add the bias.

    Args:
        x_tr (np.array): raw training features, shape (N_tr, 321)
        x_val (np.array): raw validation features, shape (N_val, 321)
        names (list): the 321 column names (preprocessing.load_feature_names)

    Returns:
        tx_tr (np.array): training matrix, standardized, bias column first
        tx_val (np.array): validation matrix, same columns, with the training statistics
    """
    params = preprocessing.fit_preprocessing(x_tr, names)
    tx_tr  = preprocessing.add_bias(preprocessing.transform(x_tr, params))
    tx_val = preprocessing.add_bias(preprocessing.transform(x_val, params))
    return tx_tr, tx_val

**Tests of the metric on hand-made arrays** (expected values computed by hand).

In [ ]:
y_true_test = np.array([1, 0, 1, 0])
print(f1_score(y_true_test, np.array([1, 0, 1, 0])))   # 1.0    perfect
print(f1_score(y_true_test, np.array([0, 0, 0, 0])))   # 0.0    predicts nobody
print(f1_score(y_true_test, np.array([1, 1, 1, 1])))   # 0.667  TP=2, FP=2, FN=0
print(f1_score(y_true_test, np.array([1, 0, 0, 0])))   # 0.667  TP=1, FP=0, FN=1

print(best_threshold(np.array([0, 0, 1, 1]), np.array([0.1, 0.4, 0.35, 0.8])))   # (0.11, 0.8)

## Step 0: load the data

`helpers.load_csv_data` reads the CSV files with `np.genfromtxt`, which takes about 1.5 minutes.
The first cell is run **once** to cache the arrays as `.npy` files in `data/` (ignored by git);
afterwards the second cell reloads them in under a second.

The labels come as {-1, 1}: logistic regression needs **{0, 1}** (1 = heart disease).

**Checks:** 328,135 training rows and 109,379 test rows with 321 features; about 8.8% positives
(heavily imbalanced classes).

In [ ]:
## ONE TIME ONLY: read the CSV files and save the numpy arrays

# x_train, x_test, y_train, train_ids, test_ids = helpers.load_csv_data(data_path)

# np.save("../data/x_train.npy", x_train)
# np.save("../data/x_test.npy", x_test)
# np.save("../data/y_train.npy", y_train)
# np.save("../data/train_ids.npy", train_ids)
# np.save("../data/test_ids.npy", test_ids)

In [ ]:
# 0 - load the datas
x_train = np.load("../data/x_train.npy")
x_test = np.load("../data/x_test.npy")
y_train = np.load("../data/y_train.npy")
test_ids = np.load("../data/test_ids.npy")   # needed later for the submission file

y = (y_train == 1).astype(int)   # labels {-1, 1} -> {0, 1}

names = preprocessing.load_feature_names(data_path)   # the 321 column names, needed by the preprocessing

## dim check
print(x_train.shape, y.shape, x_test.shape, test_ids.shape)   # (328135, 321) (328135,) (109379, 321) (109379,)
print(np.mean(y))                                             # ~0.088

## Step 1: 50k-row subsample

**Why a subsample?** While developing, we run the pipeline many times (bugs, parameters). On 50k rows a
run takes seconds instead of minutes, and the preprocessing needs much less memory (on all 328k rows the
matrix alone is ~1 GB). The validation part still has ~900 positives, enough for a reasonably stable F1.
The best settings will be confirmed on the full data, and the final model is always trained on all rows.

**Seed:** one random generator with a fixed seed, so the subsample and the split are the same at every
run (reproducible).

In [ ]:
rng = np.random.default_rng(42)

# 1 - from the loaded data - take a 50k rows subsample
x_sub, y_sub = subsample(x_train, y, 50_000, rng)
print(x_sub.shape, y_sub.shape, y_sub.mean())   # (50000, 321) (50000,) ~0.088

## Step 2: split 80% training / 20% validation

The split is done on the **raw** data, before any preprocessing: the validation part must stay
completely unseen until we evaluate.

In [ ]:
# 2 - from the subsample split the data - 80% training / 20% validation (preprocess after and test the model on it)
x_tr, y_tr, x_val, y_val = split_data(x_sub, y_sub, 0.8, rng)
print(x_tr.shape, x_val.shape, y_tr.mean(), y_val.mean())   # (40000, 321) (10000, 321) ~0.088 ~0.088

## Step 3: preprocessing

### 3a. Training part: fit + transform + bias

1. `fit_preprocessing` learns every cleaning statistic on the **training part only**: answer codes, kept
   columns, categories, medians, clip bounds, mean/std.
2. `transform` applies them: a standardized matrix without NaN.
3. `add_bias` adds a column of 1s on the left. It comes **after** standardization: standardizing a
   constant column would turn it into 0s.

**Checks:** no NaN; every feature has mean ≈ 0 and std ≈ 1; the bias column is all 1s. On a 50k
subsample there can be slightly fewer columns than on the full data (349): rare categories missing from
the subsample get no one-hot column.

In [ ]:
# 3 - preprocess the training data - fit + transform + biais
params = preprocessing.fit_preprocessing(x_tr, names)          # statistics learned on the training part only
tx_tr = preprocessing.add_bias(preprocessing.transform(x_tr, params))

## checks
col_mean = tx_tr[:, 1:].mean(axis=0)   # column by column, without the bias column
col_std  = tx_tr[:, 1:].std(axis=0)
print(tx_tr.shape)
print(np.abs(col_mean).max())          # should be ~0 (e.g. 1e-14)
print(col_std.min(), col_std.max())    # should both be ~1
print(np.all(tx_tr[:, 0] == 1))        # bias column is all 1s -> True
print(np.isnan(tx_tr).sum())           # should be 0

### 3b. Validation part: transform + bias (no fit)

The validation part is transformed with the **training statistics** (`params`). Fitting on it would leak
information from the rows we evaluate on and make the score optimistic.

**Checks:** same number of columns as the training matrix, no NaN. Means and stds are close to 0 and 1
but not exactly, since they were computed on the training part.

In [ ]:
# 3 - preprocess the validation data with the fit of train data - transform + biais
tx_val = preprocessing.add_bias(preprocessing.transform(x_val, params))

## checks
print(tx_val.shape, tx_val.shape[1] == tx_tr.shape[1])   # same columns -> True
print(np.isnan(tx_val).sum())                            # should be 0

## Step 4: train L2-regularized logistic regression

| Parameter | Value | Why |
|---|---|---|
| `lambda_` | 1e-4 | Almost no regularization for a first run, to tune later. **1e-4, not 1e4**: 1e4 makes the training diverge. |
| `initial_w` | zeros | One weight per column; zero = probability 0.5 for everyone. |
| `max_iters` | 500 | To check: does the loss still change with 1000? |
| `gamma` | 0.1 | Step size; if the loss becomes `nan`, lower it. |

The returned loss is the mean logistic loss, without the penalty term.

In [ ]:
# 4 - train the model using tx_train and y_train
lambda_ = 1e-4     # really small (almost no reg) -- tune it later
initial_w = np.zeros((tx_tr.shape[1])) # standards
max_iters = 500
gamma = 0.1 # step size

w, loss = implementations.reg_logistic_regression(y_tr, tx_tr, lambda_, initial_w, max_iters, gamma)
print("loss:", loss)   # ~0.22: finite, so the training converged

## Step 5: prediction on the validation part + analysis

1. Scores `tx_val @ w` → probabilities with the sigmoid.
2. `best_threshold` picks the threshold with the best F1.

**First result:** loss ≈ 0.22; mean probability ≈ 0.10, close to the positive rate (well calibrated on
average); best threshold ≈ 0.25, well below 0.5 as expected with imbalanced classes; **F1 ≈ 0.45**.
One split only, so about ± 0.02 of noise: cross-validation will give a more reliable number.

In [ ]:
# 5 - prediction on validation data (we ve just trained) + analyse
scores = 1 / (1 + np.exp(-(tx_val @ w)))   # sigmoid: probabilities in [0, 1]
print("scores min/mean/max:", scores.min(), scores.mean(), scores.max())
print("y_val:", np.unique(y_val, return_counts=True))

best_t, best_f1 = best_threshold(y_val, scores)
print("best threshold:", best_t, " validation F1:", best_f1)

## Next: stratified k-fold cross-validation (to do)

- `build_k_indices`: k folds with ~9% positives each.
- `cross_validate`: for each fold, `prepare_features` → train → probabilities → `best_threshold`;
  return F1 mean ± std and the mean threshold.
- Check: each fold has ~8.8% positives, and the F1 std is small.

In [ ]:
# TODO cross-validation

## Next: experiments (to do)

Change one thing at a time, log every run in the experiment log.
1. γ and number of iterations (convergence)
2. λ grid for regularized logistic regression
3. Baselines: least squares, ridge
4. Preprocessing variants (e.g. `max_missing`)
5. Confirm the best configurations on the full data

In [ ]:
# TODO experiments